# FXS / 03f — Take-profit at +15 pips, no stop loss, time exit kept (EURUSD-01, EURUSD-02, USDJPY-01, USDJPY-02, 2016 → 2026)

Question (Iván, 2026-11-10): is the edge only **timing** (a small average drift over a fixed window) or a **real move** large enough to reach a consistent +15 pips?

Condition checked first (2026-11-10): 2026 trade swings were of the same size as 2016–2025 (EURUSD-02 st. dev. 16.5 pips vs 15.9–29.2; USDJPY-01 16.1 vs 7.5–23.7; USDJPY-02 18.0 vs 5.3–15.8), so the test is run.

| Version | Exit |
|---|---|
| `BASE` | scheduled end time only |
| `TP15` | **+15 pips take-profit**, otherwise the scheduled end time; **no stop loss** |

Plus two diagnostics (reported, not a rule choice):
- **Excursions:** in each `BASE` trade, how often the price went **+15 pips in favour** vs **−15 pips against** at any minute of the window. A real directional move shows clearly more favourable than adverse touches; pure timing noise shows about the same.
- **Take-profit ladder:** +10, +15, +20, +25, +30 pips, to see whether any level holds across periods.

Fills, costs and account as 03d/03e (minute bid/ask highs and lows; a minute that could hit both a stop and the take-profit counts against you; take-profit filled at the level, or at a better open). Periods: 2016–2020 (in sample), 2021–2025, 2026 (seen). Descriptive.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

E0 = 6_000.0
LEVERAGES = [1, 2, 3, 5, 10]
COMMISSION_PER_LOT = 3.0          # USD per lot, round trip
SPREAD_FACTOR = 0.9               # x Dukascopy spread
IDS = ["EURUSD-01", "EURUSD-02", "USDJPY-01", "USDJPY-02"]
FROZEN = {"EURUSD-01": ("UTC", 75, 195, 1), "EURUSD-02": ("NY", 660, 900, -1),
          "USDJPY-01": ("UTC", 55, 115, -1), "USDJPY-02": ("UTC", 25, 55, 1)}
WINDOWS = ((dt.date(2016, 1, 1), dt.date(2020, 12, 31), "discovery"), (dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"),
           (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout"))
PERIODS = {"2016-2020 (in sample)": (2016, 2020), "2021-2025": (2021, 2025), "2026": (2026, 2026)}

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs03f_")
    os.environ.pop("DUKASCOPY_DIR", None)
    LEVERAGES = [1, 5]
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
TP_LADDER = [10.0, 15.0, 20.0, 25.0, 30.0]
VARIANTS = {"BASE": (F.NO_STOP, True), "TP15": (F.take_profit_only(15.0), True)}
LADDER = {f"TP{int(t)}": (F.take_profit_only(t), True) for t in TP_LADDER}
BRANCH = Path.cwd().parent
OUT = Path.cwd() / ("03f_synthetic" if SYNTHETIC else "03f"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220, "display.max_columns", 40)

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(("EURUSD", "USDJPY")):
        F.write_synthetic_candles(base, p, dt.date(2015, 12, 28), dt.date(2026, 11, 6), seed=11 + i, drift_bps=-3.0 if p == "EURUSD" else 0.0,
                                  drift_ny=(4, 8), price=150.0 if p.endswith("JPY") else 1.10)
    cand = pd.DataFrame([{"id": k, "pair": k[:6], "clock": c, "start": s, "end": e, "direction": d} for k, (c, s, e, d) in FROZEN.items()])
else:
    cand = F.load_frozen_candidates(BRANCH).query("id in @IDS").reset_index(drop=True)
for _, r in cand.iterrows():
    assert (r["clock"], int(r["start"]), int(r["end"]), int(r["direction"])) == FROZEN[r["id"]], f"{r['id']} is not the frozen rule"
cand = cand.set_index("id").loc[IDS].reset_index()
display(cand[["id", "pair", "clock", "start", "end", "direction"]].assign(
    rule=[f"{'buy' if d > 0 else 'sell'} {F.slot_label(c, s)} -> {F.slot_label(c, e)}" for c, s, e, d in zip(cand["clock"], cand["start"], cand["end"], cand["direction"])]))

## 1. Minute data (bid/ask open-high-low-close), 2015-12-31 → 2026-11-06

In [ ]:
PAIRS = sorted(set(cand["pair"]))
minutes, days = {}, {}
for p in PAIRS:
    parts = [F.load_minutes_ohlc(p, a, b, st, cfg.min_file_share) for a, b, st in WINDOWS]
    m = pd.concat(parts)
    minutes[p] = F.scale_spread(m[~m.index.duplicated(keep="first")].sort_index(), SPREAD_FACTOR)
    for c in set(cand.loc[cand["pair"] == p, "clock"]):
        days[(p, c)] = np.concatenate([F.build_panel(F.load_minutes(p, a, b, st, min_file_share=cfg.min_file_share), p, c, a, b, cfg.min_coverage).dates
                                       for a, b, st in WINDOWS])
    print(p, len(minutes[p]), "minutes", minutes[p].index[0], "->", minutes[p].index[-1])

## 2. Every trade, both ways (and the take-profit ladder)

In [ ]:
trades = []
for _, r in cand.iterrows():
    for v, (rule, time_exit) in VARIANTS.items():
        t = F.run_structure(minutes[r["pair"]], r["pair"], r["clock"], int(r["start"]), int(r["end"]), int(r["direction"]),
                            days[(r["pair"], r["clock"])], rule, time_exit, COMMISSION_PER_LOT)
        trades.append(t.assign(structure=r["id"], variant=v))
    for v, (rule, time_exit) in LADDER.items():
        if v in VARIANTS:
            continue
        t = F.run_structure(minutes[r["pair"]], r["pair"], r["clock"], int(r["start"]), int(r["end"]), int(r["direction"]),
                            days[(r["pair"], r["clock"])], rule, time_exit, COMMISSION_PER_LOT)
        trades.append(t.assign(structure=r["id"], variant=v))
trades = pd.concat(trades, ignore_index=True)
ladder = trades.copy()
trades = trades[trades["variant"].isin(list(VARIANTS))].reset_index(drop=True)
trades["year"] = pd.to_datetime(trades["date"]).dt.year
trades.to_parquet(OUT / "03f_trades.parquet")
print(trades.groupby(["variant", "structure"]).size().unstack())

## 3. Per structure: pips, win rate, how trades end, how long they last

In [ ]:
def per_structure(t):
    reasons = t["reason"].value_counts(normalize=True)
    return pd.Series({"trades": len(t), "net pips/trade": t["net_pips"].mean(), "win rate": (t["net_pips"] > 0).mean(),
                      "avg win pips": t.loc[t["net_pips"] > 0, "net_pips"].mean(), "avg loss pips": t.loc[t["net_pips"] <= 0, "net_pips"].mean(),
                      "% take-profit": reasons.get("tp", 0.0),
                      "% time / data end": reasons.get("time", 0.0) + reasons.get("end_of_data", 0.0),
                      "median minutes held": t["minutes"].median(), "max hours held": t["minutes"].max() / 60,
                      "worst trade pips": t["net_pips"].min(), "best trade pips": t["net_pips"].max()})
tab = trades.groupby(["structure", "variant"]).apply(per_structure)
display(tab.round(2)); tab.to_csv(OUT / "03f_per_structure.csv")

## 3b. Net pips per trade by period

In [ ]:
trades["period"] = None
for name, (a, b) in PERIODS.items():
    trades.loc[trades["year"].between(a, b), "period"] = name
per = trades.pivot_table(index=["structure", "variant"], columns="period", values="net_pips", aggfunc="mean")[list(PERIODS)]
wins = trades.assign(win=trades["net_pips"] > 0).pivot_table(index=["structure", "variant"], columns="period", values="win", aggfunc="mean")[list(PERIODS)]
display(per.round(2)); display((100 * wins).round(1).add_suffix(" win %"))
per.to_csv(OUT / "03f_period_pips.csv")

## 3c. Timing or a real move? Favourable vs adverse excursions in BASE trades

Share of trades that touched **+15 pips in favour** and **−15 pips against** at some minute of the window (after the spread at entry), by period; and the median best and worst point.

In [ ]:
b = trades[trades["variant"] == "BASE"].copy()
b["fav15"], b["adv15"] = b["best_pips"] >= 15, b["worst_pips"] <= -15
exc = b.groupby(["structure", "period"]).agg(trades=("net_pips", "size"), touched_plus15=("fav15", "mean"), touched_minus15=("adv15", "mean"),
                                             median_best=("best_pips", "median"), median_worst=("worst_pips", "median"))
exc["plus15 minus minus15 (pp)"] = 100 * (exc["touched_plus15"] - exc["touched_minus15"])
exc[["touched_plus15", "touched_minus15"]] *= 100
exc = exc.reindex(pd.MultiIndex.from_product([IDS, list(PERIODS)], names=["structure", "period"]))
display(exc.round(1)); exc.to_csv(OUT / "03f_excursions.csv")

## 3d. Take-profit ladder: net pips per trade and hit rate, by period (reported, not a rule choice)

In [ ]:
ladder["period"] = None
for name, (a, b_) in PERIODS.items():
    ladder.loc[pd.to_datetime(ladder["date"]).dt.year.between(a, b_), "period"] = name
lad = ladder.pivot_table(index=["structure", "variant"], columns="period", values="net_pips", aggfunc="mean")[list(PERIODS)]
hit = ladder.assign(h=ladder["reason"] == "tp").pivot_table(index=["structure", "variant"], columns="period", values="h", aggfunc="mean")[list(PERIODS)]
order = ["BASE"] + [f"TP{int(t)}" for t in TP_LADDER]
lad = lad.reindex(pd.MultiIndex.from_product([IDS, order]))
hit = (100 * hit).reindex(pd.MultiIndex.from_product([IDS, order]))
display(lad.round(2)); display(hit.round(1).add_suffix(" TP hit %"))
lad.to_csv(OUT / "03f_ladder_pips.csv"); hit.to_csv(OUT / "03f_ladder_hits.csv")

## 4. Net pips per trade, year by year

In [ ]:
yr = trades.pivot_table(index="year", columns=["structure", "variant"], values="net_pips", aggfunc="mean")
display(yr.round(2)); yr.to_csv(OUT / "03f_yearly_pips.csv")
fig, axes = plt.subplots(1, len(IDS), figsize=(5.2 * len(IDS), 3.6), sharey=True)
for ax, i in zip(axes, IDS):
    yr[i].plot.bar(ax=ax, width=0.8); ax.axhline(0, color="black", lw=0.5); ax.set_title(i); ax.set_xlabel("")
axes[0].set_ylabel("net pips per trade")
plt.tight_layout(); plt.savefig(OUT / "03f_yearly_pips.png", dpi=110); plt.show()

## 5. The compounding account: all four structures together, from $6,000

Equity is realised at each trade's exit.

In [ ]:
rows, curves = [], {}
for v in VARIANTS:
    tv = trades[trades["variant"] == v]
    for L in LEVERAGES:
        acc = F.simulate_events(tv, L, E0)
        curves[(v, L)] = acc
        rows.append({"variant": v, "L each": L, **F.account_stats(acc, E0)})
res = pd.DataFrame(rows)
show = res.copy()
for c in ("cagr", "ann_vol", "max_drawdown", "worst_year", "best_year"):
    show[c] = (100 * show[c]).round(1).astype(str) + "%"
show["final_equity"] = show["final_equity"].round(0)
display(show); res.to_csv(OUT / "03f_account.csv", index=False)
fig, axes = plt.subplots(1, len(VARIANTS), figsize=(13, 4.2), sharey=True)
for ax, v in zip(axes, VARIANTS):
    for L in LEVERAGES:
        ax.plot(curves[(v, L)]["equity"].clip(lower=1), label=f"{L}x each")
    ax.axhline(E0, color="grey", lw=0.6)
    for b in ("2021-01-01", "2026-01-01"):
        ax.axvline(pd.Timestamp(b), color="black", ls=":", lw=0.8)
    ax.set_yscale("log"); ax.set_title(v); ax.legend()
axes[0].set_ylabel("equity USD (log)")
plt.tight_layout(); plt.savefig(OUT / "03f_equity.png", dpi=110); plt.show()

## 5b. Annual return within each period (separate $6,000 accounts started at each period's first day)

In [ ]:
pr = []
for v in VARIANTS:
    for name, (a, b) in PERIODS.items():
        tv = trades[(trades["variant"] == v) & trades["year"].between(a, b)]
        for L in LEVERAGES:
            st = F.account_stats(F.simulate_events(tv, L, E0), E0)
            pr.append({"variant": v, "period": name, "L": L, "cagr %": round(100 * st["cagr"], 1), "max dd %": round(100 * st["max_drawdown"], 1)})
pr = pd.DataFrame(pr)
display(pr.pivot_table(index=["variant", "L"], columns="period", values=["cagr %", "max dd %"])[[("cagr %", p) for p in PERIODS] + [("max dd %", p) for p in PERIODS]])
pr.to_csv(OUT / "03f_period_account.csv", index=False)

## 6. Account return year by year (%)

In [ ]:
def yearly_returns(acc):
    y = acc["equity"].resample("YE").last()
    r = y.pct_change(); r.iloc[0] = y.iloc[0] / E0 - 1
    r.index = r.index.year
    return r
yrs = pd.DataFrame({(v, f"{L}x"): yearly_returns(curves[(v, L)]) for v in VARIANTS for L in LEVERAGES})
display((100 * yrs).round(1)); yrs.to_csv(OUT / "03f_yearly_returns.csv")

## 7. The same accounts without EURUSD-01 (the 03b set)

EURUSD-01 failed its 2021–2025 confirmation; this shows what including it costs.

In [ ]:
rows3 = []
for v in VARIANTS:
    tv = trades[(trades["variant"] == v) & (trades["structure"] != "EURUSD-01")]
    for L in LEVERAGES:
        st = F.account_stats(F.simulate_events(tv, L, E0), E0)
        rows3.append({"variant": v, "L each": L, "final_equity": round(st["final_equity"]), "cagr %": round(100 * st["cagr"], 1),
                      "max drawdown %": round(100 * st["max_drawdown"], 1), "worst year %": round(100 * st["worst_year"], 1)})
display(pd.DataFrame(rows3)); pd.DataFrame(rows3).to_csv(OUT / "03f_account_without_EURUSD01.csv", index=False)

In [ ]:
summary = {"study": "FXS03f", "descriptive": True, "synthetic": SYNTHETIC, "structures": IDS, "equity0": E0,
           "take_profit_pips": 15.0, "ladder": TP_LADDER,
           "commission_usd_per_lot_rt": COMMISSION_PER_LOT, "spread_factor": SPREAD_FACTOR,
           "per_structure": tab.round(4).reset_index().to_dict(orient="records"), "account": res.to_dict(orient="records"),
           "note": "Take-profit chosen after seeing the data; hindsight structure set; 2026 already seen."}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")